# Decision Consistency Evaluation — STL-10

STL-10: 96×96 images, 10 classes. `torchvision` handles the download automatically (~2.5 GB on first run).

Same methodology as CIFAR: 4 models, 6 non-adversarial perturbations, 3 metrics. Extra analyses:
- α-ablation (rankings stable across α ∈ {0.3, 0.4, 0.5, 0.6, 0.7})
- Pairwise Wilcoxon significance tests
- Accuracy vs Consistency scatter — correctly classified samples can still be brittle
- Quantitative Grad-CAM cosine similarity: stable vs label-flipped predictions

**Update the paths in the Configuration cell before running.**

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
STL10_ROOT     = r'E:\stl10'
PREV_CIFAR_CSV = r'E:\decision_consistency_cifar_outputs\tables\summary_table.csv'
OUTPUT_DIR     = r'E:\decision_consistency_stl10_outputs'

NUM_SAMPLES = 1000
SEED        = 42
ALPHA       = 0.5

In [ ]:
import os, json, random, io, csv, gc
from collections import defaultdict

import numpy as np
from scipy.stats import wilcoxon, pearsonr, spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms, datasets
from PIL import Image
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

for sub in ['tables','figures/consistency','figures/gradcam',
            'figures/perclass','figures/ablation','figures/correlation']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
stl10_raw = datasets.STL10(root=STL10_ROOT, split='test', download=True)
STL10_CLASSES = ['airplane','bird','car','cat','deer',
                 'dog','horse','monkey','ship','truck']
X_stl = stl10_raw.data
y_stl = np.array(stl10_raw.labels)
print(f'STL-10: {X_stl.shape}  classes: {STL10_CLASSES}')

In [ ]:
def load_models(device):
    zoo = [('ResNet-18', models.resnet18), ('ResNet-50', models.resnet50),
           ('VGG-16', models.vgg16), ('MobileNetV2', models.mobilenet_v2)]
    md = {}
    for name, fn in zoo:
        print(f'Loading {name}...', end=' ', flush=True)
        md[name] = fn(pretrained=True).eval().to(device); print('OK')
    return md

MODELS = load_models(device)
model_names = list(MODELS.keys())

In [ ]:
# STL-10 is 96×96 — the 224 resize is better justified here than on 32×32 CIFAR
preprocess = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def prepare(t):
    return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)

def apply_jpeg(t, q=75):
    buf = io.BytesIO()
    transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf, format='JPEG', quality=q)
    buf.seek(0); return transforms.ToTensor()(Image.open(buf))

def generate_perturbations(t):
    noise = 0.01 * torch.randn_like(t)
    return {'rotation':   TF.rotate(t, angle=2),
            'brightness': TF.adjust_brightness(t, brightness_factor=1.1),
            'noise':      torch.clamp(t + noise, 0.0, 1.0),
            'contrast':   TF.adjust_contrast(t, contrast_factor=1.1),
            'blur':       TF.gaussian_blur(t, kernel_size=3, sigma=0.5),
            'jpeg':       apply_jpeg(t)}

def infer(model, t):
    with torch.no_grad():
        probs = F.softmax(model(prepare(t)), dim=1)
    return probs.argmax(dim=1).item(), probs.max().item()

def compute_metrics(orig_pred, orig_conf, pert_results, alpha=ALPHA):
    K = len(pert_results)
    S = sum(1 for p,_ in pert_results.values() if p == orig_pred) / K
    D = sum(abs(c-orig_conf)/max(orig_conf,1-orig_conf) for _,c in pert_results.values()) / K
    return {'Label_Stability':S, 'Confidence_Deviation':D,
            'Consistency_Score': alpha*S + (1-alpha)*(1-D)}

print('Pipeline ready.')

In [ ]:
X_torch = torch.tensor(X_stl, dtype=torch.float32) / 255.0
n_eval  = min(NUM_SAMPLES, len(y_stl))
results = {'STL-10': {}}

for model_name, model in MODELS.items():
    per_sample = []
    for idx in tqdm(range(n_eval), desc=f'STL-10/{model_name}'):
        img = X_torch[idx]
        orig_pred, orig_conf = infer(model, img)
        pert_results = {k: infer(model, v) for k, v in generate_perturbations(img).items()}
        m = compute_metrics(orig_pred, orig_conf, pert_results)
        m['true_label'] = int(y_stl[idx])
        m['class_name'] = STL10_CLASSES[int(y_stl[idx])]
        m['orig_pred']  = orig_pred
        m['orig_conf']  = orig_conf
        per_sample.append(m)
    results['STL-10'][model_name] = per_sample
    ls = np.mean([m['Label_Stability']     for m in per_sample])
    cd = np.mean([m['Confidence_Deviation'] for m in per_sample])
    cs = np.mean([m['Consistency_Score']    for m in per_sample])
    print(f'  [STL-10][{model_name}]  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')

print('Evaluation complete.')

In [ ]:
# Summary CSV
header = ['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation',
          'Avg_Consistency_Score','Std_Consistency_Score']
rows = []
print(f'{"Model":<14} {"LS":>6} {"CD":>7} {"CS":>7} {"±CS":>7}')
print('-' * 44)
for mn in model_names:
    ms  = results['STL-10'][mn]
    ls  = np.mean([m['Label_Stability']      for m in ms])
    cd  = np.mean([m['Confidence_Deviation']  for m in ms])
    cs  = np.mean([m['Consistency_Score']     for m in ms])
    std = np.std( [m['Consistency_Score']     for m in ms])
    print(f'{mn:<14} {ls:6.3f} {cd:7.3f} {cs:7.3f} {std:7.3f}')
    rows.append({'Dataset':'STL-10','Model':mn,
                 'Avg_Label_Stability':round(ls,4),
                 'Avg_Confidence_Deviation':round(cd,4),
                 'Avg_Consistency_Score':round(cs,4),
                 'Std_Consistency_Score':round(std,4)})
csv_path = os.path.join(OUTPUT_DIR, 'tables', 'summary_table_stl10.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=header); w.writeheader(); w.writerows(rows)
print('Saved:', csv_path)

In [ ]:
# Pairwise Wilcoxon significance tests
cs_arrays = {mn: np.array([m['Consistency_Score'] for m in results['STL-10'][mn]])
             for mn in model_names}
print('=== Wilcoxon Tests (STL-10) ===')
print(f'{"Pair":<28} {"W":>10} {"p":>10} {"sig":>6}')
print('-' * 58)
sig_rows = []
for i, mn1 in enumerate(model_names):
    for mn2 in model_names[i+1:]:
        w, p = wilcoxon(cs_arrays[mn1], cs_arrays[mn2])
        pair = f'{mn1} vs {mn2}'
        sig  = '***' if p < 0.05 else 'ns'
        print(f'{pair:<28} {w:>10.2f} {p:>10.4f} {sig:>6}')
        sig_rows.append({'Pair':pair,'W':round(w,2),'p_value':round(p,6),'Significant':p<0.05})
csv_path = os.path.join(OUTPUT_DIR, 'tables', 'wilcoxon_tests_stl10.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['Pair','W','p_value','Significant'])
    w.writeheader(); w.writerows(sig_rows)
print('Saved:', csv_path)

In [ ]:
# α-ablation
alpha_values = [0.3, 0.4, 0.5, 0.6, 0.7]
fig, ax = plt.subplots(figsize=(8, 4))
for mn, marker in zip(model_names, ['o','s','^','D']):
    ms   = results['STL-10'][mn]
    ls_a = np.array([m['Label_Stability']     for m in ms])
    cd_a = np.array([m['Confidence_Deviation'] for m in ms])
    vals = [float(np.mean(a*ls_a + (1-a)*(1-cd_a))) for a in alpha_values]
    ax.plot(alpha_values, vals, marker=marker, label=mn, linewidth=2, markersize=7)
ax.set_xlabel('α (weight of Label Stability)', fontsize=11)
ax.set_ylabel('Average Consistency Score', fontsize=11)
ax.set_title('α-Ablation Study — STL-10\nModel rankings stable across all α', fontsize=11)
ax.set_xticks(alpha_values); ax.set_ylim(0, 1)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'ablation', 'alpha_ablation_stl10.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# Key figure: correctly classified samples can still be brittle
for mn in model_names:
    ms       = results['STL-10'][mn]
    accuracy = [1 if m['orig_pred'] == m['true_label'] else 0 for m in ms]
    cs_vals  = [m['Consistency_Score'] for m in ms]
    fig, ax  = plt.subplots(figsize=(5, 4))
    jitter   = np.random.uniform(-0.04, 0.04, size=len(accuracy))
    ax.scatter(np.array(accuracy, dtype=float) + jitter, cs_vals,
               alpha=0.25, s=8, color='steelblue')
    for lv, color, lbl in [(1,'seagreen','Correct'), (0,'tomato','Wrong')]:
        subset = [cs for acc, cs in zip(accuracy, cs_vals) if acc == lv]
        if subset:
            ax.axhline(np.mean(subset), color=color, linestyle='--', lw=1.5,
                       label=f'{lbl} mean={np.mean(subset):.2f}')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Incorrect', 'Correct'])
    ax.set_ylim(0, 1); ax.set_ylabel('Consistency Score')
    ax.set_title(f'{mn} — Accuracy vs Consistency\n'
                 'Correct predictions can still be brittle', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    fname = f'acc_vs_cs_STL10_{mn}.png'.replace('-','_').replace(' ','_')
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'consistency', fname),
                dpi=300, bbox_inches='tight'); plt.close()
    print('Saved:', fname)

In [ ]:
# Cross-model bar chart
avg_ls = [np.mean([m['Label_Stability']      for m in results['STL-10'][mn]]) for mn in model_names]
avg_cs = [np.mean([m['Consistency_Score']     for m in results['STL-10'][mn]]) for mn in model_names]
avg_cd = [np.mean([m['Confidence_Deviation']  for m in results['STL-10'][mn]]) for mn in model_names]
x, w = np.arange(len(model_names)), 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x-w, avg_ls, w, label='Label Stability',      color='steelblue')
ax.bar(x,   avg_cs, w, label='Consistency Score',    color='darkorange')
ax.bar(x+w, avg_cd, w, label='Confidence Deviation', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0,1); ax.set_ylabel('Score')
ax.set_title('Cross-Model Decision Consistency — STL-10', fontsize=12)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'consistency', 'crossmodel_STL10.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# Per-class consistency
for mn in model_names:
    ms           = results['STL-10'][mn]
    class_scores = defaultdict(list)
    for m in ms: class_scores[m['class_name']].append(m['Consistency_Score'])
    sorted_cls = sorted({cls: np.mean(s) for cls,s in class_scores.items()}.items(),
                        key=lambda x: x[1])
    cls_names = [c for c,_ in sorted_cls]
    cls_vals  = [v for _,v in sorted_cls]
    fig, ax   = plt.subplots(figsize=(8, 4))
    ax.bar(cls_names, cls_vals,
           color=['tomato' if v<0.5 else 'steelblue' for v in cls_vals],
           edgecolor='white')
    ax.set_xticklabels(cls_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0,1); ax.set_ylabel('Avg Consistency Score')
    ax.set_title(f'Per-Class Consistency — STL-10 / {mn}\n'
                 '(red = below 0.5 threshold)', fontsize=10)
    ax.axhline(0.5, color='gray', linestyle='--', lw=1)
    plt.tight_layout()
    fname = f'perclass_STL10_{mn}.png'.replace('-','_').replace(' ','_')
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'perclass', fname),
                dpi=300, bbox_inches='tight'); plt.close()
    print('Saved:', fname)